# Matrix Multiplication

For loops are slow, but matrix operations are fast.

Suppose we have $C = A B$. We must have that $A$ has $m$ columns and $B$ has $m$ rows. Then $C_{ij} = \sum_{k=1}^ma_{ik}b_{kj} = A_i(B^T)_j$.

A single entry requires $2m-1$ operations.

If $A,B$ are both $n\times n$, then we require $2n^3-n^2=O(n^3)$ operations.



In [1]:
import numpy as np
import matplotlib.pyplot as plt

## Vectors and Matrices

In [2]:
# vectors: list of numbers
v = np.array([1,2,3,4])
v

array([1, 2, 3, 4])

In [3]:
w = np.array([1,-1,1,-1])

In [4]:
wv

NameError: name 'wv' is not defined

In [ ]:
# vector addition
v+w

In [ ]:
# entrywise multiplication
v*w

In [ ]:
# dot product
v.dot(w)

In [ ]:
# vector willed with zeros
np.zeros(10)

In [ ]:
# vector filled with ones
np.ones(10)

In [ ]:
# random vector (entries are from a standard normal distribution)
np.random.randn(10)

In [ ]:
# matrices = list of lists

A = np.array([[1,2,3],[4,5,6],[7,8,9]])
A

In [ ]:
np.zeros((4,3))

In [ ]:
B = np.ones((3,3))
B

In [ ]:
np.random.randn(4,4)

In [ ]:
# matrix addition
A+B

In [ ]:
# entrywise multiplication
A*B

In [ ]:
# matrix multiplication
A.dot(B)

In [ ]:
A@B # bad alternative

In [ ]:
# matrix times a vector

v = np.array([1,2,3])
A.dot(v)

### Matrix Multiplication

In [ ]:
def mult_3loops(A,B):
    m,n = A.shape # rows, columns
    q,p = B.shape 
    
    # initialize C
    C = np.zeros((m,p))
    
    for i in range(m):
        for j in range(p):
            for k in range(n):
                C[i,j] += A[i,k]*B[k,j]
            
    return C

In [ ]:
def mult_2loops(A,B):
    m,n = A.shape # rows, columns
    q,p = B.shape 
    
    # initialize C
    C = np.zeros((m,p))
    for i in range(m):
        for j in range(n):
            C[i,j] = A[i,:].dot(B[:,j])
    return C

In [ ]:
def mult_1loop(A,B):
    m,n = A.shape # rows, columns
    q,p = B.shape 
    # initialize C
    C = np.zeros((m,p))
    for j in range(p):
        C[:,j] = A.dot(B[:,j])
    return C

In [ ]:
A = np.random.randn(200,200)
B = np.random.randn(200,200)

In [ ]:
%%timeit
mult_3loops(A,B)

In [ ]:
%%timeit
mult_2loops(A,B)

In [ ]:
%%timeit
mult_1loop(A,B)

In [ ]:
%%timeit
A.dot(B)

Why does this take forever? Type checks. They're like vibe checks but for types

In [ ]:
import time

# Number of operations = O(n^3) ??

n_list = [1000,2000,4000,8000]


# initialize time vector
t = []

for n in n_list:
    A = np.random.randn(n,n)
    B = np.random.randn(n,n)
    # start the timer
    start = time.time()
    C = A.dot(B)
    end = time.time() # stop the timer
    
    t.append(end-start)
    
    
    


In [ ]:
plt.loglog(n_list,t,'o-')

In [ ]:
# slope: should be about 3
(np.log(t[3])-np.log(t[0]))/(np.log(n_list[3])-np.log(n_list[0]))

### Matrix Multiplication is Associative

But where should I but the parentheses?

Suppose $A=n\times 1, B = 1\times n, C = n\times 1$

$(AB)C$ is a product of an $n\times n$ matrix with an $n\times 1$ vector. Making the matrix and doing the multiplication each needs about n^2 operations.

On the other hand, $A(BC)$ is a product of a vector with a scalar and uses about $3n$ operations.

In [5]:
A = np.random.randn(1000,1)
B = np.random.randn(1,1000)
C = np.random.randn(1000,1)

In [7]:
%%timeit

D = (A.dot(B)).dot(C)

447 µs ± 15.8 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [8]:
%%timeit

D = A.dot(B.dot(C))

1.48 µs ± 9.63 ns per loop (mean ± std. dev. of 7 runs, 1000000 loops each)


# Gaussian Elimination and the LU Factorization

Goal: to solve the linear system $Ax=b$

How do we solve? Gaussian Elimination, which equivalent factoring $A$ as $LU$.

In [10]:
A = np.array([[2,4,-2],[4,9,-3],[-2,-3,7]])
A

array([[ 2,  4, -2],
       [ 4,  9, -3],
       [-2, -3,  7]])

In [29]:
def my_lu_3loops(A):
    # size of A
    m,n = A.shape
    # check if A is square
    if m!=n:
        print('Warning: A must be square')
        return
    # initialize L and U
    L = np.identity(n)
    U = A.copy()
    # Gaussian Elimination steps
    for i in range(n):
        for j in range(i+1,n):
            L[j,i] = U[j,i]/U[i,i]
            # row operations
            U[j,i]=0
            for k in range(i+1,n):
                U[j,k] = U[j,k] - L[j,i]*U[i,k]
    return L,U
    

In [13]:
my_lu_3loops(A)

(array([[ 1.,  0.,  0.],
        [ 2.,  1.,  0.],
        [-1.,  1.,  1.]]),
 array([[ 2,  4, -2],
        [ 0,  1,  1],
        [ 0,  0,  4]]))

In [30]:
def my_lu_2loops(A):
    # size of A
    m,n = A.shape
    # check if A is square
    if m!=n:
        print('Warning: A must be square')
        return
    # initialize L and U
    L = np.identity(n)
    U = A.copy()
    # Gaussian Elimination steps
    for i in range(n):
        for j in range(i+1,n):
            L[j,i] = U[j,i]/U[i,i]
            # row operations
            U[j,i]=0
            U[j,i+1:n] = U[j,i+1:n]-L[j,i]*U[i,i+1:n]
    return L,U

In [18]:
my_lu_2loops(A)

(array([[ 1.,  0.,  0.],
        [ 2.,  1.,  0.],
        [-1.,  1.,  1.]]),
 array([[ 2,  4, -2],
        [ 0,  1,  1],
        [ 0,  0,  4]]))

In [ ]:
L[i+1:n,i] = U[i+1:n,i]/U[i,i]

U[i+1:n,i+1:n] = U[i+1:n,i+1:n] - L[i+1:n,i].dot(U[i,])

In [31]:
def my_lu(A):
    # size of A
    m,n = A.shape
    # check if A is square
    if m!=n:
        print('Warning: A must be square')
        return
    # initialize L and U
    L = np.identity(n)
    U = A.copy()
    # Gaussian Elimination steps
    for i in range(n):
        L[i+1:n,i]= U[i+1:n,i]/U[i,i]
        U[i+1:n,i]=0
        U[i+1:n,i+1:n] = U[i+1:n,i+1:n] -np.outer(L[i+1:n,i],U[i,i+1:n])
    return L,U

In [24]:
n = 1000
A = np.random.randn(n,n)

In [32]:
%%timeit
my_lu_3loops(A)

KeyboardInterrupt: 

In [ ]:
%%timeit
my_lu_2loops(A)

In [ ]:
%%timeit
my_lu(A)